# Validación del EKF con azúcar total en LAB022–024

Se aplica el protocolo del EKF 13 a los estados \(X,N,S,E,C_d,I\), con \(S=G+F\) e \(I\) como acumulador de CO₂. Se usa el modelo S con solubilidad variable del EKF 06, con sus parámetros químicos y de transferencia de gas fijos. Se conserva \(R=0{,}00218\;(\mathrm{g/(L\,h)})^2\), la temperatura registrada, la media de flujo de los últimos 30 minutos y la regla de dos ventanas consecutivas del 13. La corrida principal usa el CSV interpolado. La segunda deja el hueco sin observación. El punto interpolado usa una lectura futura y el ensayo que lo incluye es retrospectivo.

El azúcar total y el YAN medidos en la inoculación inicializan la química. La primera muestra de biomasa viable fija el inicio del filtro a los 15 minutos. Se mantienen el etanol inicial (10,010194 g/L) y los ceros iniciales de CO₂ disuelto y acumulado. Las muestras posteriores de biomasa, YAN y azúcar total se reservan para evaluar el filtro.

En la hora de nutrición registrada para cada lote se suma al YAN el aporte nominal de 1,0 g de SpringFerm Xtrem (SFX) y 0,4 g de fosfato diamónico (FDA), según las masas confirmadas por el usuario. Se usan 0,10 g de YAN equivalente por g de SFX según la ficha de Fermentis y 2 × 14,007 / 132,1 g N por g de FDA, a partir de la fórmula y la masa molar publicadas por la OIV. El modelo representa ambos aportes como un único salto instantáneo y los divide por el volumen remanente a la hora del pulso. [Ficha de Fermentis](https://atpgroup.com/wp-content/uploads/2021/01/SpringFermXtrem.pdf) · [Monografía de la OIV](https://www.oiv.int/node/3844/download/pdf). La comparación se reporta también por separado antes y después de la nutrición. Para el volumen se usa la carga registrada y se descuentan solo las retiradas de 50 mL documentadas.

## Modelo S y covarianzas

Para \(P_0\), al sumar G y F se obtiene \(\operatorname{var}(S_0)=12{,}5+37{,}5+2(-12{,}5)=25\;(\mathrm{g/L})^2\). Los demás términos iniciales se toman del 13. Para \(Q\) se usa el ruido de proceso por flujos del EKF-S 06, con \(Q=\Delta t\,L D L^\top\) evaluado al inicio de cada paso. La covarianza cinética del 13 se estimó para \(\mu_0,\beta_{G,0},\beta_{F,0}\) en el modelo G/F. Se usa el mapeo de flujos del 06 para el modelo de azúcar total.

In [ ]:
from pathlib import Path
import ast
import json
import sys

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import scipy
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/wine_fermentation").is_dir())
EKF = ROOT / "notebooks/EKF"
print(f"Kernel {Path(sys.prefix).name} · SciPy {scipy.__version__}")

# Read fixed coefficients and functions without executing calibration or other replays.
source06 = EKF / "06_zenteno_single_sugar_variable_solubility.source.ipynb"
model_nb = nbformat.read(source06, as_version=4)
trees = [(cell.source, ast.parse(cell.source)) for cell in model_nb.cells if cell.cell_type == "code"]
required = {"theta", "PS", "GAS_PARAMETERS", "PCO2_BAR"}
parameters = {}
for _, tree in trees:
    for node in ast.walk(tree):
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if isinstance(target, ast.Name) and target.id in required:
                    assert target.id not in parameters
                    parameters[target.id] = ast.literal_eval(node.value)
assert parameters.keys() == required
base = parameters["theta"]
PS = parameters["PS"]
YXS = 1 / (1 / base["Yxg"] + 1 / base["Yxf"])
KLA, GAIN = parameters["GAS_PARAMETERS"]["S"]
SCALE = np.array([.3, .25, 160., 15., .2, 1.])
scale_outer = np.outer(SCALE, SCALE)
STATES = ("X", "N", "S", "E", "Cd", "I")
R = 0.00218

model_scope = dict(np=np, theta=base, PS=PS, YXS=YXS, KLA=KLA, GAIN=GAIN,
                   PCO2_BAR=parameters["PCO2_BAR"], SCALE_EKF=SCALE)
for names, sources in (
    ({"co2_saturation", "gas_exchange"}, trees),
    ({"rates_ekf", "jacobian", "noise_matrix"}, [(s, tree) for s, tree in trees if "PS={'mu0'" in s]),
):
    definitions = {node.name: ast.get_source_segment(s, node)
                   for s, tree in sources for node in tree.body
                   if isinstance(node, ast.FunctionDef) and node.name in names}
    assert definitions.keys() == names
    for definition in definitions.values():
        exec(compile(definition, str(source06), "exec"), model_scope)

rates_ekf = model_scope["rates_ekf"]
jacobian = model_scope["jacobian"]
noise_matrix = model_scope["noise_matrix"]
display(pd.Series({**PS, **{name: base[name] for name in ("Kn0", "Kd0", "m0")},
                   "Yxs": YXS, "kLa": KLA, "gain": GAIN, "R": R}, name="Valor fijo"))


In [ ]:
def transition(points, start, end, data, S0):
    """Advance scaled trajectories with the one-minute RK4 scheme of EKF 13."""
    w = points * SCALE
    n = max(1, int(np.ceil((end - start) * 60 - 1e-7)))
    dt = (end - start) / n
    available = data["available"]
    temperatures = data["temperature"]
    for j in range(n):
        t = start + j * dt
        times = np.array([t, t + dt / 2, t + dt])
        indices = np.searchsorted(available, times + 1e-12, side="right") - 1
        if indices[0] < 0:
            raise ValueError("No hay temperatura causal al comienzo del intervalo")
        T0, Tm, T1 = temperatures[indices]
        k1 = rates_ekf(w, t, T0, S0)
        k2 = rates_ekf(w + dt * k1 / 2, t + dt / 2, Tm, S0)
        k3 = rates_ekf(w + dt * k2 / 2, t + dt / 2, Tm, S0)
        k4 = rates_ekf(w + dt * k3, t + dt, T1, S0)
        w = np.maximum(w + dt * (k1 + 2 * k2 + 2 * k3 + k4) / 6, 0.)
    return w / SCALE


model_scope["transition"] = transition


In [ ]:
def replay(data):
    grid = data["grid"]
    z = data["initial"] / SCALE
    P = data["P0"] / scale_outer
    states = np.empty((len(grid), len(z)))
    covariance = np.empty((len(grid), len(z), len(z)))
    diag = np.full((len(grid), 7), np.nan)
    injected = np.zeros_like(covariance)
    for i, end in enumerate(grid):
        if i:
            start = grid[i - 1]
            index = np.searchsorted(data["available"], start + 1e-12, side="right") - 1
            assert index >= 0
            T = data["temperature"][index]
            Q = (end - start) * noise_matrix(z * SCALE, T, data["S0"])
            z, J = jacobian(z, start, end, data, data["S0"])
            injected[i] = Q * scale_outer
            P = J @ P @ J.T + Q
            P = (P + P.T) / 2

        z[1] += data["doses"][i] / SCALE[1]
        if end > 0 and abs(end / .5 - round(end / .5)) < 1e-8:
            predicted = z[5] / .5
            innovation = data["y"][i] - predicted
            variance = 4 * P[5, 5] + R
            assert variance > 0
            clips = 0
            if data["update"][i]:
                K = 2 * P[:, 5] / variance
                updated = z + K * innovation
                clips = np.count_nonzero(updated[:5] < 0)
                B = np.eye(len(z))
                B[:, 5] -= 2 * K
                P = B @ P @ B.T + np.outer(K, K) * R
                P = (P + P.T) / 2
                z = np.maximum(updated, 0)
            diag[i] = [predicted, innovation, variance, innovation**2 / variance, clips,
                       data["update"][i], data["valid"][i]]
            z[5] = 0.
            P[5, :] = P[:, 5] = 0.
        states[i] = z * SCALE
        covariance[i] = P * scale_outer
    assert np.isfinite(states).all() and np.isfinite(covariance).all()
    assert np.linalg.eigvalsh(injected).min() > -1e-9
    assert np.linalg.eigvalsh(covariance).min() > -1e-9
    np.testing.assert_array_equal(diag[:, 5] == 1, data["update"])
    return states, covariance, diag

## Muestras y señales de LAB022–024

La lectura y selección de muestras sigue la preparación de datos de LAB022–024. Para comparar biomasa, la concentración viable Oculyze en millones de células/mL se multiplica por el factor histórico de 0,03 g/L por millón de células/mL.

In [ ]:
import csv
from datetime import datetime

%matplotlib inline

cwd = Path.cwd().resolve()
data_dir = next(
    path / 'data/original/natural/lab022_024'
    for path in (cwd, *cwd.parents)
    if (path / 'data/original/natural/lab022_024').is_dir()
)
labs = ['LAB022', 'LAB023', 'LAB024']
workbook = data_dir / 'lab022-024MedicionesOffline_estandar.xlsx'
experiments = pd.read_excel(workbook, sheet_name='Experimentos').set_index('CodigoLab').loc[labs]
inoculation = experiments['FechaHora_Inoculacion']

offline = (
    pd.read_excel(workbook, sheet_name='MedicionesOffline')
    .rename(columns={'CodigoLab': 'lab', 'Muestra': 'sample'})
    .loc[lambda frame: frame['lab'].isin(labs)]
    .copy()
)
offline['sampled_at'] = [
    datetime.combine(date.date(), time)
    for date, time in zip(offline['Fecha'], offline['Hora'])
]
nutrition = (
    offline.loc[offline['Observaciones'].eq('Hora de nutrición')]
    .set_index('lab')['sampled_at']
    .loc[labs]
)
offline = offline.dropna(subset='sample').copy()
offline['sample'] = offline['sample'].replace(
    {**{number: f'{number:02d}' for number in range(1, 7)}, 'POST NUT': 'POSTNUT'}
)
offline = offline.sort_values(['lab', 'sampled_at'])

def elapsed_hours(frame):
    return (frame['sampled_at'] - frame['lab'].map(inoculation)).dt.total_seconds() / 3600

offline['hours'] = elapsed_hours(offline)
sample_times = offline[['lab', 'sample', 'sampled_at']]
post_times = sample_times.loc[sample_times['sample'].eq('POSTNUT')].set_index('lab')['sampled_at']
assert post_times.eq(pd.Timestamp('2026-11-09 11:30')).all()
assert post_times.gt(nutrition).all()


y15_sources = {
    'EXPAuto(2026-11-02 11-57).txt': {'3': '01', '02': '02', '03': '03', '04': '04'},
    'ONLINE(26-11-09 09-52)_1.txt': {'05': '05', '06': '06', 'POSTNUT': 'POSTNUT'},
}
y15_columns = ['source_label', 'assay', 'station', 'reading', 'unit', 'assayed_at']
assays = ['PAN', 'AMMONIA', 'YAN', 'GLUCOSE-320', 'GLU-FRU-320']
y15_frames = []
for filename, samples in y15_sources.items():
    labels = {
        f'{lab}-{suffix}': (lab, sample)
        for lab in labs
        for suffix, sample in samples.items()
    }
    raw = pd.read_csv(
        data_dir / filename, sep='\t', header=None, names=y15_columns,
        dtype={'reading': str}, encoding='utf-8-sig',
    )
    keep = raw['source_label'].isin(labels) & raw['assay'].isin(assays)
    selected = raw.loc[keep].drop_duplicates().copy()
    selected[['lab', 'sample']] = selected['source_label'].map(labels).tolist()
    selected['assayed_at'] = pd.to_datetime(selected['assayed_at'], format='%d-%m-%Y %H:%M:%S')
    selected['reading'] = pd.to_numeric(selected['reading'].str.replace(',', '.', regex=False))
    selected['source_file'] = filename
    y15_frames.append(selected)

y15 = pd.concat(y15_frames, ignore_index=True).merge(
    sample_times, on=['lab', 'sample'], how='left', validate='many_to_one'
)
assert y15['sampled_at'].notna().all()
assert y15.loc[y15['sample'].eq('01'), 'sampled_at'].eq(
    y15.loc[y15['sample'].eq('01'), 'lab'].map(inoculation)
).all()
assert y15.loc[y15['sample'].eq('POSTNUT'), 'assayed_at'].dt.date.eq(
    datetime(2026, 11, 9).date()
).all()

# Pivot also checks that each lab, sample, and assay has one value.
chemistry = (
    y15.pivot(index=['lab', 'sample', 'sampled_at'], columns='assay', values='reading')
    .reset_index()
    .sort_values(['lab', 'sampled_at'])
)
chemistry['hours'] = elapsed_hours(chemistry)

# Prefer instrument-reported YAN where available and calculate the remaining values.
ammonia_n_fraction = 0.8225
chemistry['YAN (calculated)'] = chemistry['PAN'] + ammonia_n_fraction * chemistry['AMMONIA']
chemistry['YAN for plot'] = chemistry['YAN'].fillna(chemistry['YAN (calculated)'])

expected_samples = {'01', '02', '03', '04', '05', '06', 'POSTNUT'}
assert chemistry.groupby('lab')['sample'].agg(set).eq(expected_samples).all()
assert chemistry['YAN for plot'].notna().all()
assert chemistry.loc[chemistry['sample'].eq('POSTNUT'), 'YAN'].notna().all()


oculyze_frames = []
for lab in labs:
    filename = data_dir / f'oculyze_{lab.lower()}.csv'
    # Each CSV record is wrapped in a single quoted field.
    with filename.open(encoding='utf-8-sig', newline='') as file:
        records = [row[0] for row in csv.reader(file) if row]
    measurements = pd.DataFrame(csv.DictReader(records))
    labels = measurements['Name']
    if lab == 'LAB023':
        labels = labels.replace({'LAB024-05': 'LAB023-05'})
    measurements[['lab', 'sample']] = labels.str.rsplit('-', n=1, expand=True)
    assert measurements['lab'].eq(lab).all()
    measurements['sampled_at'] = pd.to_datetime(
        measurements['Description'].str.strip(), format='%d/%m/%y %H:%M'
    )
    for column in ['Concentration', 'Viability', 'Density']:
        measurements[column] = pd.to_numeric(measurements[column])
    oculyze_frames.append(measurements)

oculyze = pd.concat(oculyze_frames, ignore_index=True).sort_values(['lab', 'sampled_at'])
assert oculyze.groupby('lab')['sample'].agg(set).eq(expected_samples - {'POSTNUT'}).all()
density_check = oculyze.merge(
    offline[['lab', 'sample', 'Densidad_g_L']], on=['lab', 'sample'], validate='one_to_one'
)
numbered_later = density_check.loc[density_check['sample'].ne('01')]
assert numbered_later['Density'].eq(numbered_later['Densidad_g_L']).all()
oculyze['hours'] = elapsed_hours(oculyze)
oculyze['Viable concentration'] = oculyze['Concentration'] * oculyze['Viability'] / 100

In [ ]:
instrument = json.loads((ROOT / "configs/co2_instrument.json").read_text(encoding="utf-8"))
conversion = 44.0095 * 60 * instrument["co2_correction_factor"] / (1000 * 24.16)
imputed = pd.read_csv(ROOT / "data/processed/lab022_024_co2_30min_imputed.csv",
                      parse_dates=["window_end_local"]).set_index("lab", verify_integrity=True)
assert set(imputed.index) == set(labs)
assert imputed["source"].eq("linear_interpolation_for_plot").all()

reactor_to_lab = pd.Series(experiments.index, index=experiments["Fermentador"])
def backup(filename, column):
    frame = pd.read_csv(data_dir / filename, usecols=["timestamp", "fermentador", column],
                        parse_dates=["timestamp"]).rename(columns={"timestamp": "sampled_at"})
    frame["lab"] = frame["fermentador"].map(reactor_to_lab)
    frame["hours"] = elapsed_hours(frame)
    lower = -1 / 60 if column == "T" else 0.
    frame = frame.loc[frame["hours"].ge(lower)].sort_values(["lab", "sampled_at"])
    assert frame["lab"].notna().all() and not frame.duplicated(["lab", "sampled_at"]).any()
    assert np.isfinite(frame[column]).all()
    return frame

gas = backup("backup_global_co2.csv", "flow_sccm")
temperature = backup("backup_global_temperatura.csv", "T")
nutrition_h = (nutrition - inoculation).dt.total_seconds() / 3600
sfx_mass_g, fda_mass_g = 1.0, 0.4
sfx_yan_equivalent_g_per_g = .10
fda_n_g_per_g = 2 * 14.007 / 132.1
pulse_n_g = sfx_mass_g * sfx_yan_equivalent_g_per_g + fda_mass_g * fda_n_g_per_g
biomass = oculyze.assign(X_g_L=oculyze["Viable concentration"] * .03)
X0 = biomass.loc[biomass["sample"].eq("01")].set_index("lab")
chem0 = chemistry.loc[chemistry["sample"].eq("01")].set_index("lab")
assert X0["hours"].eq(.25).all() and chem0["hours"].eq(0.).all()
assert offline["Vol_Residual_mL"].isna().all()
withdrawals = offline.loc[offline["Observaciones"].eq("se quitan 50ml")]
volumes_initial = experiments["Volumen_Inicial_L"].str.removesuffix("L").astype(float)
assert withdrawals.groupby("lab").size().eq(4).all()

def prepare_lab(lab):
    filled = imputed.loc[lab]
    assert filled["fermentador"] == experiments.at[lab, "Fermentador"]
    fill_h = float(filled["window_end_h"])
    assert filled["window_end_local"] == inoculation[lab] + pd.Timedelta(hours=fill_h)
    removed_at = withdrawals.loc[withdrawals["lab"].eq(lab), "hours"].to_numpy()
    def volume(hours):
        return volumes_initial[lab] - .05 * np.searchsorted(removed_at, hours, side="right")

    raw = gas.loc[gas["lab"].eq(lab) & gas["hours"].gt(0)].copy()
    raw["bin"] = np.ceil(raw["hours"] / .5 - 1e-10).astype(int)
    raw["flow_mass"] = raw["flow_sccm"] * conversion / volume(raw["hours"])
    last_hour = np.floor(raw["hours"].max() / .5) * .5
    rows = []
    for index in range(1, int(last_hour / .5) + 1):
        end = index * .5
        window = raw.loc[raw["bin"].eq(index)]
        max_gap = np.diff(np.r_[end - .5, window["hours"], end]).max()
        valid = len(window) >= 30 and max_gap <= 2 / 60 + 1e-10
        rows.append((end, window["flow_sccm"].mean() if valid else np.nan,
                     window["flow_mass"].mean() if valid else np.nan, valid))
    observed = pd.DataFrame(rows, columns=["hours", "sccm", "measured", "valid"]).set_index("hours")
    assert observed.index[~observed["valid"]].tolist() == [fill_h]
    endpoints = [filled["left_window_end_h"], filled["right_window_end_h"]]
    endpoint_flow = [filled["left_mean_sccm"], filled["right_mean_sccm"]]
    np.testing.assert_allclose(observed.loc[endpoints, "sccm"], endpoint_flow, rtol=0, atol=1e-8)
    np.testing.assert_allclose(filled["flow_mean_sccm"], np.interp(fill_h, endpoints, endpoint_flow),
                               rtol=0, atol=1e-8)
    assert volume(fill_h - .5) == volume(fill_h)
    augmented = observed.copy()
    augmented.loc[fill_h, ["sccm", "measured", "valid"]] = [
        filled["flow_mean_sccm"], filled["flow_mean_sccm"] * conversion / volume(fill_h), True]

    rt = temperature.loc[temperature["lab"].eq(lab)].copy()
    rt["minute"] = np.floor(rt["hours"] * 60 + 1e-10).astype(int)
    tm = rt.groupby("minute")["T"].median()
    available = (tm.index.to_numpy() + 1) / 60
    assert available[0] <= X0.at[lab, "hours"] and available[-1] >= last_hour

    ini = np.array([X0.at[lab, "X_g_L"], chem0.at[lab, "YAN for plot"] / 1000,
                    chem0.at[lab, "GLU-FRU-320"],
                    10.010194, 0., 0.])
    assert np.isfinite(ini).all() and (ini >= 0).all()
    P0 = np.diag([max(.03, .2 * ini[0])**2, .015**2, 25., 4., 0., 0.])
    start = X0.at[lab, "hours"]
    pulse_h = round(nutrition_h[lab], 10)
    assert start < pulse_h < last_hour
    grid = np.unique(np.round(np.r_[np.arange(start, last_hour, 1 / 12), observed.index, pulse_h], 10))
    pulse_index = np.searchsorted(grid, pulse_h)
    assert grid[pulse_index] == pulse_h
    volume_at_pulse = volume(pulse_h)
    doses = np.zeros(len(grid))
    doses[pulse_index] = pulse_n_g / volume_at_pulse
    indices = np.searchsorted(grid, observed.index)
    np.testing.assert_array_equal(grid[indices], observed.index)
    settings = dict(grid=grid, doses=doses, initial=ini, P0=P0, available=available,
                    temperature=tm.to_numpy(float), S0=ini[2])
    cases = {}
    for name, sensor in (("Interpolado", augmented), ("Sin rellenar", observed)):
        usable = sensor["valid"] & sensor["measured"].ge(.05)
        assimilate = usable & usable.shift(fill_value=False)
        y = np.full(len(grid), np.nan)
        valid, update = np.zeros(len(grid), bool), np.zeros(len(grid), bool)
        y[indices], valid[indices], update[indices] = sensor["measured"], sensor["valid"], assimilate
        assert not update[indices[0]]
        cases[name] = dict(settings, y=y, valid=valid, update=update)
    added = grid[cases["Interpolado"]["update"] & ~cases["Sin rellenar"]["update"]]
    assert set(added) <= {fill_h, fill_h + .5}
    audit = {"LAB": lab, "X inicial [g/L]": ini[0], "YAN inicial [mg/L]": ini[1] * 1000,
             "S inicial [g/L]": ini[2],
             "Volumen a 22 h [L]": volume(fill_h), "Horizonte [h]": last_hour,
             "CO₂ interpolado [g/(L h)]": augmented.at[fill_h, "measured"],
             "Correcciones con CSV": int(cases["Interpolado"]["update"].sum()),
             "Correcciones sin rellenar": int(cases["Sin rellenar"]["update"].sum()),
             "Horas de corrección añadidas": ", ".join(f"{hour:g}" for hour in added) or "—"}
    return cases, audit

data = {}
audit_rows = []
for lab in labs:
    data[lab], audit = prepare_lab(lab)
    audit_rows.append(audit)
audit_table = pd.DataFrame(audit_rows).set_index("LAB")
display(audit_table.round(4))
pulse_table = pd.DataFrame({
    "Hora local": nutrition,
    "Hora desde inoculación [h]": nutrition_h,
    "Volumen [L]": {lab: pulse_n_g / data[lab]["Interpolado"]["doses"].sum() for lab in labs},
    "Pulso nominal [mg N/L]": {lab: 1000 * data[lab]["Interpolado"]["doses"].sum() for lab in labs},
})
display(pulse_table.round(3))

## Comparación con las muestras

Se calcula el RMSE solo con las muestras posteriores a las usadas para inicializar. Si una muestra cae entre pasos, se avanza el modelo desde el último estado disponible hasta la hora de esa muestra. Las dos corridas se puntúan en las mismas horas.

Para CO₂ se compara la predicción del promedio de 30 minutos, antes de corregir con la medición, con todas las ventanas reales válidas a partir de 1 h. El RMSE de ambas corridas usa esas mismas ventanas y excluye el dato ficticio de las 22 h. El etanol y el CO₂ disuelto solo cuentan con estimaciones del filtro.

In [ ]:
results = {}
for lab in labs:
    print(f"Corriendo {lab}", flush=True)
    results[lab] = {}
    for name, inputs in data[lab].items():
        results[lab][name] = replay(inputs)

panels = (
    ("CO2", "CO₂ flow · 30-min mean", "g/(L h)", None, None, 1, r"$\widehat{y}_{k|k-1}$"),
    ("X", "Biomass", "g/L", biomass, "X_g_L", 1, r"$\widehat{X}_{k|k}$"),
    ("N", "YAN", "mg N/L", chemistry, "YAN for plot", 1000, r"$\widehat{N}_{k|k}$"),
    ("S", "Total sugar", "g/L", chemistry, "GLU-FRU-320", 1, r"$\widehat{S}_{k|k}$"),
    ("E", "Ethanol", "g/L", None, None, 1, r"$\widehat{E}_{k|k}$"),
    ("Cd", "Dissolved CO₂", "g/L", None, None, 1, r"$\widehat{C}_{d,k|k}$"),
)

def state_series(states, variable):
    return states[..., STATES.index(variable)]


def state_at(sample_hour, inputs, states):
    grid = inputs["grid"]
    j = np.searchsorted(grid, sample_hour + 1e-9, side="right") - 1
    assert j >= 0 and sample_hour <= grid[-1]
    if abs(sample_hour - grid[j]) <= 1e-9:
        return states[j]
    predicted = transition((states[j] / SCALE)[None, :], grid[j], sample_hour, inputs, inputs["S0"])
    return predicted[0] * SCALE

rows = []
for lab in labs:
    for variable, _, unit, frame, column, factor, _ in panels:
        if frame is None:
            continue
        later = frame.loc[frame["lab"].eq(lab) & frame["sample"].ne("01")].dropna(subset=[column])
        for _, sample in later.iterrows():
            hour = sample["hours"]
            row = {"LAB": lab, "Estado": variable, "Unidad": unit, "Muestra": sample["sample"],
                   "Hora": hour, "Periodo": "Antes" if hour < nutrition_h[lab] else "Después",
                   "Medido": sample[column]}
            for name, inputs in data[lab].items():
                states = state_at(hour, inputs, results[lab][name][0])
                row[name] = factor * state_series(states, variable)
            rows.append(row)

sample_comparison = pd.DataFrame(rows)
counts = sample_comparison.groupby(["LAB", "Estado"]).size().unstack()
assert counts[["X", "N"]].eq({"X": 5, "N": 6}).all().all()
assert counts["S"].eq(5).all()
assert np.isfinite(sample_comparison[["Medido", "Interpolado", "Sin rellenar"]]).all().all()

def summarize(group_columns, samples=sample_comparison):
    rows = []
    for key, group in samples.groupby(group_columns, sort=False):
        row = dict(zip(group_columns, key))
        row["n"] = len(group)
        for name in ("Interpolado", "Sin rellenar"):
            row[name] = np.sqrt(np.mean((group[name] - group["Medido"])**2))
        row["Diferencia de RMSE"] = row["Interpolado"] - row["Sin rellenar"]
        rows.append(row)
    return pd.DataFrame(rows).set_index(group_columns)

rmse_table = summarize(["LAB", "Estado", "Unidad"])
yan_by_period = summarize(["LAB", "Periodo"], sample_comparison.loc[sample_comparison["Estado"].eq("N")])
co2_rows = []
for lab in labs:
    real = data[lab]["Sin rellenar"]
    grid = real["grid"]
    # The first window is only partially integrated because the filter starts at 0.25 h.
    observed = real["valid"] & (grid > .5)
    co2_rows.append(pd.DataFrame({
        "LAB": lab, "Medido": real["y"][observed],
        **{name: result[2][observed, 0] for name, result in results[lab].items()},
    }))
co2_comparison = pd.concat(co2_rows, ignore_index=True)
assert np.isfinite(co2_comparison[["Medido", "Interpolado", "Sin rellenar"]]).all().all()
co2_rmse = summarize(["LAB"], co2_comparison)
display(rmse_table.round(5))
display(co2_rmse.round(5))
display(yan_by_period.round(3))
display(sample_comparison.round(4))

## Flujo de CO₂ y estados estimados

La curva azul muestra la corrida con el CSV interpolado. En el panel de flujo, el círculo naranja indica la ventana de 30 minutos interpolada a las 22 h. La predicción de flujo se obtiene dividiendo el incremento de \(I\) en la ventana por 0,5 h. La línea vertical señala la hora en que el filtro aplica el aporte de nitrógeno.

Los seis gráficos muestran la trayectoria completa hasta las 190 h. Los puntos de laboratorio aparecen en sus horas de muestreo.

In [ ]:
def legend_inside(ax, error_label):
    handles, labels = ax.get_legend_handles_labels()
    entries = dict(zip(labels, handles))
    assert len(labels) == len(entries)
    legend = ax.legend(entries.values(), entries.keys(), loc="best",
                       fontsize=8, framealpha=.85, title=error_label, title_fontsize=8)
    if error_label:
        legend.get_title().set_ha("left")


for variable, label, unit, frame, column, factor, symbol in panels:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
    for ax, lab in zip(axes, labs):
        filled = data[lab]["Interpolado"]
        real = data[lab]["Sin rellenar"]
        grid = filled["grid"]
        states, _, diagnostics = results[lab]["Interpolado"]
        end = grid[-1] + 2
        score = None
        if variable == "CO2":
            windows = np.isfinite(diagnostics[:, 0])
            predicted = windows & (grid > .5)
            ax.plot(grid[predicted], diagnostics[predicted, 0], color="#0072B2", lw=1.5, label=symbol)
            ax.plot(grid[windows], real["y"][windows], color="black", lw=1, label="Samples")
            synthetic = filled["valid"] & ~real["valid"]
            ax.scatter(grid[synthetic], filled["y"][synthetic], facecolors="none",
                       edgecolors="#D55E00", s=34, label="Interpolated", zorder=5)
            score = co2_rmse.at[lab, "Interpolado"]
        else:
            if frame is not None:
                samples = frame.loc[frame["lab"].eq(lab)].dropna(subset=[column])
                ax.scatter(samples["hours"], samples[column], color="black", s=23,
                           label="Samples", zorder=5)
                score = rmse_table.at[(lab, variable, unit), "Interpolado"]
            estimate = factor * state_series(states, variable)
            ax.plot(grid, estimate, color="#0072B2", lw=1.5, label=symbol)
        ax.axvline(nutrition_h[lab], color="gray", ls=":", lw=.8, label="N pulse")
        ax.set(title=lab, xlabel="Time [h]", xlim=(-2., end))
        ax.grid(alpha=.2)
        legend_inside(ax, f"RMSE = {score:.3f} {unit}" if score is not None else None)
    axes[0].set_ylim(bottom=0.)
    axis_label = "CO₂ flow" if variable == "CO2" else label
    axes[0].set_ylabel(f"{axis_label} [{unit}]")
    fig.suptitle(label, y=.99)
    fig.tight_layout(pad=.7, w_pad=.9)
    plt.show()